In [2]:
import pyspark
print(pyspark.__version__)

4.1.1


In [9]:
# ============================================================
# CAPSDAC SAFE SPARK SETUP + DATA LOADING
# De-identified local mode by default
# Secure ABFSS mode only on issued laptop / approved environment
# ============================================================

import os
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ============================================================
# 0) RUN MODE
# ============================================================

# Keep False for GitHub / personal laptop / de-identified demo.
# Change to True ONLY on issued laptop with approved CDE/Azure access.
USE_SECURE_ABFSS = False


# ============================================================
# 1) PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

LOCAL_SAMPLE_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Child_April_deidentified_sample.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2) SECURE ABFSS PATHS
# Use only inside approved CDE laptop/environment
# ============================================================

CHLD_PATH = "abfss://capsdacreporting-parquet@capsdacdatalakestg.dfs.core.windows.net/Production/Reporting/CHLD/"
FAMI_PATH = "abfss://capsdacreporting-parquet@capsdacdatalakestg.dfs.core.windows.net/Production/Reporting/FAMI/"
ENRL_PATH = "abfss://capsdacreporting-parquet@capsdacdatalakestg.dfs.core.windows.net/Production/Reporting/ENRL/"
CLEN_PATH = "abfss://capsdacreporting-parquet@capsdacdatalakestg.dfs.core.windows.net/Production/Reporting/CLEN/"


# ============================================================
# 3) SPARK SESSION
# ============================================================

spark = (
    SparkSession.builder
    .appName("CAPSDAC_ML_System_Deidentified")
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")
    .config("spark.sql.parquet.int96RebaseModeInRead", "LEGACY")
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

print("Spark session started")


# ============================================================
# 4) SAFE READ FUNCTIONS
# ============================================================

def read_local_deidentified_sample(path=LOCAL_SAMPLE_PATH):
    """
    Read the safe de-identified sample CSV included in the GitHub package.
    This is the default mode for portfolio/demo use.
    """
    if not Path(path).exists():
        raise FileNotFoundError(
            f"De-identified sample file not found: {path}\n"
            "Expected file: data/raw/Child_April_deidentified_sample.csv"
        )

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(path))
    )

    print(f"Loaded local de-identified sample: {path}")
    print(f"Rows: {df.count():,}")
    return df


def read_secure_parquet(path):
    """
    Read secure parquet from ABFSS.
    Use only on issued laptop / approved CDE environment.

    If this fails locally with getSubject / ABFSS errors, it means:
    - Java/Spark/Hadoop environment is incompatible, or
    - Azure ABFSS credentials/connectors are unavailable, or
    - the machine is not authorized.
    """
    return (
        spark.read
        .option("datetimeRebaseMode", "LEGACY")
        .option("int96RebaseMode", "LEGACY")
        .option("mergeSchema", "false")
        .parquet(path)
    )


# ============================================================
# 5) LOAD DATA
# ============================================================

if USE_SECURE_ABFSS:
    print("Running in SECURE ABFSS mode.")
    print("Use this only on the issued laptop / approved CDE environment.")

    chld = read_secure_parquet(CHLD_PATH)
    fami = read_secure_parquet(FAMI_PATH)
    enrl = read_secure_parquet(ENRL_PATH)
    clen = read_secure_parquet(CLEN_PATH)

else:
    print("Running in SAFE LOCAL DE-IDENTIFIED mode.")
    print("No CDE warehouse or raw child-level data is accessed.")

    sample_df = read_local_deidentified_sample()

    # For local demo, use the same safe aggregate sample as working input.
    chld = sample_df
    fami = None
    enrl = None
    clen = None


# ============================================================
# 6) BASIC DATA REVIEW
# ============================================================

print("Columns:")
chld.printSchema()

print("Sample rows:")
chld.show(10, truncate=False)


# ============================================================
# 7) SAFE AGGREGATION EXAMPLE
# ============================================================

# This works with the de-identified sample.
# Adjust column names only if your secure data has different names.

if "ReportMonth" in chld.columns and "EnrollmentCount" in chld.columns:
    monthly_enrollment = (
        chld.groupBy("ReportMonth")
        .agg(F.sum("EnrollmentCount").alias("TotalEnrollment"))
        .orderBy("ReportMonth")
    )

    monthly_enrollment.show()

    monthly_enrollment_pd = monthly_enrollment.toPandas()
    monthly_enrollment_pd.to_csv(
        TABLE_DIR / "monthly_enrollment_summary.csv",
        index=False
    )

    print("Saved:", TABLE_DIR / "monthly_enrollment_summary.csv")

else:
    print("ReportMonth / EnrollmentCount columns not found. Check schema above.")


# ============================================================
# 8) COUNTY CONTRIBUTION EXAMPLE
# ============================================================

if all(c in chld.columns for c in ["ReportMonth", "CountyName", "EnrollmentCount"]):
    county_contribution = (
        chld.groupBy("ReportMonth", "CountyName")
        .agg(F.sum("EnrollmentCount").alias("CountyEnrollment"))
        .orderBy("ReportMonth", F.desc("CountyEnrollment"))
    )

    county_contribution.show()

    county_contribution_pd = county_contribution.toPandas()
    county_contribution_pd.to_csv(
        TABLE_DIR / "county_enrollment_contribution.csv",
        index=False
    )

    print("Saved:", TABLE_DIR / "county_enrollment_contribution.csv")

else:
    print("County contribution columns not found. Check schema above.")

Spark session started
Running in SAFE LOCAL DE-IDENTIFIED mode.
No CDE warehouse or raw child-level data is accessed.


FileNotFoundError: De-identified sample file not found: /Users/yuzhang/projects/Machine_learning/01_capsdac_ml_system/notebooks/data/raw/Child_April_deidentified_sample.csv
Expected file: data/raw/Child_April_deidentified_sample.csv